[back to [index.ipynb](index.ipynb)]

## Building the Michelson entity data set

- Read the raw 1879 instrument readings from `data/michelson-1879.dat`
- attach the units they were recorded in
- converts to SI
- wraps each column in an `Entity` carrying its EMMO term and description
- compute displacement $d$
- write the resulting `EntityCollection` as CSV, YAML and HDF5.
  - the content of the file is the same as `data/michelson-1879-entities.csv`



In [1]:
import mammos_entity as me
import mammos_units as u
import numpy as np
from astropy.units import imperial  # special case: needed to get to feet as unit 

datafile = "data/michelson-1879.dat"

### Reading the raw data

`data/michelson-1879.dat` is headerless and unitless: 100 observations, four columns.

| column | meaning | unit as recorded |
|---|---|---|
| 1 | rotation rate of the revolving mirror | revolutions per second |
| 2 | revolving mirror to the crosshair of the micrometer | feet |
| 3 | displacement of the returned image | turns of the micrometer screw |
| 4 | travel of the crosshair per turn of that screw | millimetres per turn |

One number the apparatus needed is not in the table at all: the separation of the
revolving and the fixed mirror, `D = 1986.23 ft`.

In [2]:
# read all 100 lines of data with 4 columns
raw = np.loadtxt(datafile)
raw[0:5, :]

array([[257.36   ,  28.672  , 114.55   ,   0.99614],
       [257.52   ,  28.655  , 114.56   ,   0.99614],
       [257.52   ,  28.647  , 114.5    ,   0.99614],
       [193.14   ,  28.647  ,  85.84   ,   0.99598],
       [193.14   ,  28.65   ,  85.89   ,   0.99598]])

In [3]:
raw.shape  # shows how many (rows, columns) of data we have

(100, 4)

### 2. Quantities, in the units the instrument used

In [4]:
# Extract arrays of data from file (metadata comes from paper)

N     = raw[:, 0] / u.s          # revolutions per second
r     = raw[:, 1] * imperial.ft  # radial distance in feet
turns = raw[:, 2]                # turns of the screw (dimensionless)
pitch = raw[:, 3] * u.mm         # millimetres per turn of screw

D = 1986.23 * imperial.ft        # mirror separation 

print(f"D     = {D:.4f}")        # print some example values
print(f"r[0]  = {r[0]:.4f}")

D     = 1986.2300 ft
r[0]  = 28.6720 ft


### 3. Convert quantities to SI

In [5]:
N_si     = N.to(u.Hz)   # rotations per second in Hertz
r_si     = r.to(u.m)    # lengths in metre
pitch_si = pitch.to(u.m)
D_si     = D.to(u.m)

print(f"D     = {D_si:.4f}")    # print some example values
print(f"r[0]  = {r_si[0]:.4f}")


D     = 605.4029 m
r[0]  = 8.7392 m


`turns` is dimensionless, no conversion needed.

### 4. Create entities from Quantities

In [6]:
N = me.Entity(
    "Frequency",
    N_si,
    description=(
        "Rotation rate of the revolving mirror, in revolutions per second, "
        "timed against a tuning fork. "
    ),
)

In [7]:
N

Entity(ontology_label='Frequency', value=array([257.36, 257.52, 257.52, 193.14, 193.14, 257.42, 257.39, 257.39,
       257.39, 257.39, 257.39, 257.29, 257.29, 257.45, 257.45, 257.45,
       257.49, 257.49, 257.49, 257.42, 257.42, 257.42, 257.42, 257.42,
       258.7 , 255.69, 257.58, 257.6 , 257.59, 257.57, 257.56, 257.36,
       257.33, 257.32, 257.32, 257.32, 257.62, 257.59, 257.58, 257.43,
       257.43, 257.43, 257.43, 257.43, 257.65, 257.65, 257.62, 257.43,
       257.43, 257.43, 257.65, 257.63, 257.62, 257.61, 257.36, 257.4 ,
       257.39, 257.39, 257.38, 257.38, 257.65, 257.64, 257.63, 257.61,
       257.6 , 257.42, 257.38, 257.37, 257.38, 257.38, 257.32, 257.33,
       257.32, 257.3 , 257.29, 257.5 , 257.49, 257.48, 257.47, 257.45,
       257.33, 257.33, 257.46, 257.44, 257.43, 257.42, 257.42, 257.42,
       193.  , 193.  , 193.  , 193.  , 257.35, 257.34, 257.28, 257.28,
       192.95, 128.63,  96.48,  64.32]), unit='Hz', description='Rotation rate of the revolving mirror, in revolutions per second, timed against a tuning fork. ')

In [8]:
r = me.Entity(
    "RadialDistance",
    r_si,
    description=(
        "Distance from the axis of the revolving mirror to the crosshair of the "
        "micrometer."
    ),
)

In [9]:
screw_turns = me.Entity(
    "AngularDisplacement",
    turns,
    description=(
        "Rotation of the micrometer screw between the setting on the slit and the "
        "setting on the deflected image. Measured in turns of the "
        "screw, not necessarily a whole number. Dimensionless."
    ),
)

In [10]:
screw_pitch = me.Entity(
    "Length",
    pitch_si,
    description=(
        "Travel of the micrometer crosshair per one full turn of the screw. "
        "Recorded in millimetres per turn. The pitch varies along the screw, hence one calibration value per observation. "
        "EMMO has no term for screw pitch; Length is the closest I could find."
    ),
)

## 5. Compute distance $d$ (derived entities) 

In [11]:
d_si = pitch_si * turns
d_si[0]

<Quantity 0.11410784 m>

In [12]:
d = me.Entity(
    "Distance",
    d_si,
    description=(
        "Separation of the slit and the deflected image, obtained as "
        "screw_turns * screw_pitch. "
    ),
)

d

Entity(ontology_label='Distance', value=array([0.11410784, 0.1141178 , 0.11405803, 0.08549492, 0.08554472,
       0.11408791, 0.11402815, 0.11401818, 0.11402815, 0.11405803,
       0.11408791, 0.11407795, 0.11409788, 0.1142971 , 0.11425726,
       0.11423734, 0.11212552, 0.11212552, 0.11213548, 0.11212552,
       0.11212552, 0.11213548, 0.11212552, 0.11213548, 0.11271324,
       0.11144814, 0.11213548, 0.11213548, 0.11211556, 0.11213548,
       0.11213548, 0.11208567, 0.11206575, 0.1120259 , 0.11203587,
       0.11205579, 0.1121554 , 0.11214544, 0.1121554 , 0.1121056 ,
       0.11207571, 0.11207571, 0.11207571, 0.11208567, 0.11220521,
       0.11220521, 0.11222513, 0.11208567, 0.11204583, 0.11205579,
       0.11223509, 0.11221517, 0.11223509, 0.11222513, 0.13271313,
       0.13273305, 0.13272309, 0.13274301, 0.13272309, 0.13272309,
       0.13279283, 0.13281275, 0.13281275, 0.13280279, 0.13280279,
       0.13271313, 0.1326932 , 0.13270316, 0.13270316, 0.1326932 ,
       0.13266331, 0.13265335, 0.13267328, 0.13266331, 0.13266331,
       0.13270316, 0.13267328, 0.13266331, 0.13266331, 0.13268324,
       0.13265335, 0.13267328, 0.13272309, 0.13270316, 0.13271313,
       0.1326932 , 0.13270316, 0.1326932 , 0.09932614, 0.09931617,
       0.09930621, 0.09930621, 0.13248398, 0.13250391, 0.13251387,
       0.13250391, 0.09905817, 0.06606535, 0.04955101, 0.0330287 ]), unit='m', description='Separation of the slit and the deflected image, obtained as screw_turns * screw_pitch. ')

## 6. Entity Collection (the whole data set)

Create a description:

In [13]:
description = f"""Michelson's 1879 determination of the speed of light: the instrument readings.

A. A. Michelson, "Experimental Determination of the Velocity of Light", Astronomical Papers of the American Ephemeris 1 (1880) 109-145. 100 observations made at the U.S. Naval Academy, Annapolis, 5 June - 2 July 1879. Transcribed via the R package loon.data (michelson_1879); see also MacKay and Oldford, Statistical Science 15(3) 254-278 (2000), doi:10.1214/ss/1009212817.

Rotating-mirror method. Light travels from the revolving mirror to a fixed mirror and back while the mirror turns; the returned beam is deflected by twice the mirror rotation and read at radial distance r as a displacement d, giving c = 8 * pi * N * D * r / d

The apparatus constant D is the separation of the revolving and the fixed mirror. It is not part of the observation table and is recorded only here: D = 1986.23 ft = {D_si.value:.4f} m

The revolving mirror rotates with frequence N. The speed of light is not stored in this file: it is derived from these readings. Note that the light travelled in air, not in vacuum."""

Turn all entities together into an *entity collection* to represent the data from the experiment.

In [14]:
experiment = me.EntityCollection(
    description,
    N=N,
    r=r,
    screw_turns=screw_turns,
    screw_pitch=screw_pitch,
    d=d,
)
experiment

EntityCollection(
    description='Michelson\'s 1879 determination of the speed of light: the instrument readings.\n\nA. A. Michelson, "Experimental Determination of the Velocity of Light", Astronomical Papers of the American Ephemeris 1 (1880) 109-145. 100 observations made at the U.S. Naval Academy, Annapolis, 5 June - 2 July 1879. Transcribed via the R package loon.data (michelson_1879); see also MacKay and Oldford, Statistical Science 15(3) 254-278 (2000), doi:10.1214/ss/1009212817.\n\nRotating-mirror method. Light travels from the revolving mirror to a fixed mirror and back while the mirror turns; the returned beam is deflected by twice the mirror rotation and read at radial distance r as a displacement d, giving c = 8 * pi * N * D * r / d\n\nThe apparatus constant D is the separation of the revolving and the fixed mirror. It is not part of the observation table and is recorded only here: D = 1986.23 ft = 605.4029 m\n\nThe revolving mirror rotates with frequence N. The speed of light is not stored in this file: it is derived from these readings. Note that the light travelled in air, not in vacuum.',
    N=Entity(ontology_label='Frequency', value=array([257.36 257.52 257.52 ... 128.63 96.48 64.32]) (shape=(100,)), unit='Hz', description='Rotation rate of the revolving mirror, in revolutions per second, timed against a tuning fork. '),
    r=Entity(ontology_label='RadialDistance', value=array([8.7392256 8.734044 8.7316056 ... 10.120884 10.120884 10.120884]) (shape=(100,)), unit='m', description='Distance from the axis of the revolving mirror to the crosshair of the micrometer.'),
    screw_turns=Entity(ontology_label='AngularDisplacement', value=array([114.55 114.56 114.5 ... 66.34 49.76 33.17]) (shape=(100,)), description='Rotation of the micrometer screw between the setting on the slit and the setting on the deflected image. Measured in turns of the screw, not necessarily a whole number. Dimensionless.'),
    screw_pitch=Entity(ontology_label='Length', value=array([0.00099614 0.00099614 0.00099614 ... 0.00099586 0.0009958 0.00099574]) (shape=(100,)), unit='m', description='Travel of the micrometer crosshair per one full turn of the screw. Recorded in millimetres per turn. The pitch varies along the screw, hence one calibration value per observation. EMMO has no term for screw pitch; Length is the closest I could find.'),
    d=Entity(ontology_label='Distance', value=array([0.11410784 0.1141178 0.11405803 ... 0.06606535 0.04955101 0.0330287]) (shape=(100,)), unit='m', description='Separation of the slit and the deflected image, obtained as screw_turns * screw_pitch. '),
)

## 7. Write data from experiment to disk

The entity collection in `data/michelson-1879-entities.csv` has been created using the above code, followed by `experiment.to_csv("data/michelson-1879-entities.csv")`.

We save the experiment here to different files so they are easier to inspect if desired.

In [15]:
experiment.to_csv("ec.csv")
experiment.to_yaml("ec.yaml")
experiment.to_hdf5("ec.h5")

In [16]:
experiment.to_dataframe(include_units=True)

,N (Hz),r (m),screw_turns,screw_pitch (m),d (m)
0,257.36,8.739226,114.55,0.000996,0.114108
1,257.52,8.734044,114.56,0.000996,0.114118
2,257.52,8.731606,114.50,0.000996,0.114058
3,193.14,8.731606,85.84,0.000996,0.085495
4,193.14,8.732520,85.89,0.000996,0.085545
...,...,...,...,...,...
95,257.28,10.153193,133.00,0.000996,0.132504
96,192.95,10.120884,99.45,0.000996,0.099058
97,128.63,10.120884,66.34,0.000996,0.066065
98,96.48,10.120884,49.76,0.000996,0.049551
